# Этап 3 Прототип нейроаналитика конкурентов

Сквозная демонстрация по принятой базе этапа 2: выбор компаний → проверка источников → сравнение → отчёт → гипотезы. Запустите все ячейки по порядку.

**По умолчанию работает без ключа, сети и платных запросов.** Сведения извлечены реальными LLM-опытами этапа 2; новый текст формируется Python по явным правилам. Полная смысловая точность 175 фактов не подтверждена. Подробный корпус — две компании; остальные семь конкурентов явно помечаются как не исследованные. Это снимок 22.10.2026, не актуализация рынка.

In [1]:
#@title Распаковать код и принятую базу
from pathlib import Path
import base64, hashlib, io, json, lzma, sys, zipfile
ROOT = Path.cwd()/"stage3_demo"
ROOT.mkdir(exist_ok=True)
PACKED = """..."""
data = base64.b64decode(PACKED)
assert hashlib.sha256(data).hexdigest() == "75216c9c6d20dad12e619708f95766d3b56b3acf215e8d10e241f151e42f9738"
with zipfile.ZipFile(io.BytesIO(lzma.decompress(data))) as archive:
    for name in archive.namelist():
        assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
sys.path.insert(0,str(ROOT/"code"))
from analyst import read, verify_inputs, run
verify_inputs(ROOT)
print("Целостность входов подтверждена. Фактов этапа 2:",len(read(ROOT/"input/facts_draft.json")["facts"]))


Целостность входов подтверждена. Фактов этапа 2: 175


## Выбор сценария

Собственная компания и восемь конкурентов уже согласованы в предыдущем этапе. Можно выбрать подмножество подтверждённых конкурентов. Для Испании демонстрация честно вернёт пустой результат. Повторный сбор сайтов — отдельная работа. Шесть критериев заданы исследователем явно; это не автоматическая полная классификация всех услуг.

In [2]:
CONFIG=read(ROOT/"config.json")
COUNTRY="Portugal"  # "Spain" — контроль отсутствия исследованного корпуса
CONFIG["country"]=COUNTRY
print("Своя компания:",CONFIG["own_company"])
print("Подтверждённые конкуренты:",CONFIG["competitors"])
print("Страна:",COUNTRY)


Своя компания: immigration_relocator
Подтверждённые конкуренты: ['migrun', 'migran', 'relotus', 'relocate_group', 'e_residence', 'dh_private', 'migrall', 'movetocascais']
Страна: Portugal


## Сравнение и отчёт

Проверяются манифест, происхождение, точная цитата и число. Цены разных составов не ранжируются. Все выводы остаются черновыми наблюдениями; три действия сформулированы как проверяемые гипотезы.

In [3]:
RESULT=run(ROOT,CONFIG)
print("Критериев:",len(RESULT["criteria"]),"доказательств:",len(RESULT["evidence"]),"гипотез:",len(RESULT["recommendations"]))
for company in RESULT["coverage"]: print(company["name"],company["facts_in_demo"],company["status"])
for row in RESULT["criteria"]:
    print("\n",row["name"],row["numeric_comparison"])
    for c in row["cells"]:
        if c["fact_ids"]: print(c["company_id"],c["amounts_eur"],c["status"])
for h in RESULT["recommendations"]: print("\n",h["id"],h["title"],"\n",h["action"],"\nПроверка:",h["experiment"],"\nМетрика:",h["metric"])


Критериев: 6 доказательств: 13 гипотез: 3
Immigration Relocator 8 limited_snapshot
Migrun 5 limited_snapshot
Migran 0 not_researched_for_selected_country
Relotus 0 not_researched_for_selected_country
Релокейт Групп 0 not_researched_for_selected_country
e-residence 0 not_researched_for_selected_country
DISCUS HOLDINGS PRIVATE CLIENTS 0 not_researched_for_selected_country
MigrAll Portugal 0 not_researched_for_selected_country
MoveToCascais 0 not_researched_for_selected_country

 Входная консультация blocked_unverified_scope_or_terms
immigration_relocator [150] observed_draft
migrun [40, 70] observed_draft

 Налоговая консультация blocked_unverified_scope_or_terms
immigration_relocator [170, 350] conflicting_published_prices

 Оформление NIF blocked_unverified_scope_or_terms
immigration_relocator [150] observed_draft

 Продление ВНЖ blocked_unverified_scope_or_terms
immigration_relocator [500] observed_draft

 Пакеты сопровождения blocked_unverified_scope_or_terms
migrun [530, 1290, 3490]

## Проверки ошибок и границ

Девять проверок включают подмену цитаты/компании/числа, конфликт цен, незнакомого конкурента, Испанию без данных, выдуманную ссылку LLM и безопасную обработку ошибки API. Транспорт API в тесте искусственный; этот тест не выдаётся за платный модельный прогон.

In [4]:
import unittest
suite=unittest.defaultTestLoader.discover(str(ROOT/"code"),pattern="test_*.py")
outcome=unittest.TextTestRunner(verbosity=1,stream=sys.stdout).run(suite)
assert outcome.wasSuccessful()
print("Проверок успешно:",outcome.testsRun)


.........
----------------------------------------------------------------------
Ran 9 tests in 4.335s

OK
Проверок успешно: 9


## Необязательная новая формулировка LLM

Не нужна для демонстрации. Включать только для отдельного исследования формулировок. Один запрос без повторов; максимум 2500 выходных токенов. Новому Colab может потребоваться разрешение доступа к секрету. Модель предварительно выбрана по пилоту этапа 2, но качество генерации рекомендаций отдельно не измерено. Ответ и настройки сохраняются; ссылки проверяются Python, смысл — человеком. Здесь нет нового сравнения цен моделей.

In [5]:
RUN_LLM=False
if RUN_LLM:
    from google.colab import userdata
    from llm_report import generate
    narration=generate(RESULT,userdata.get("PROXYAPI_KEY"),ROOT/"results/llm",CONFIG["model"])
    print("LLM создала черновик; нужна смысловая проверка.",len(narration["hypotheses"]))
else:
    print("Новых LLM-вызовов нет. Основной результат воспроизведён по реальным извлечениям этапа 2.")


Новых LLM-вызовов нет. Основной результат воспроизведён по реальным извлечениям этапа 2.


## Открыть и сохранить результат

В results/demo находятся result.json, comparison.csv, report.md, report.html и run.json. В Colab они доступны слева в панели файлов. Для скачивания архива задайте DOWNLOAD=True; выгрузка не вызывает API. Источники и даты приведены в отчёте. Клиентские и внутренние конфиденциальные данные не используются.

In [6]:
try:
    from IPython.display import HTML, display
    display(HTML((ROOT/"results/demo/report.html").read_text(encoding="utf-8")))
except ImportError:
    print("HTML сохранён; локальный Python без IPython. Откройте results/demo/report.html в браузере.")
DOWNLOAD=False
if DOWNLOAD:
    import shutil
    from google.colab import files
    zip_path=shutil.make_archive(str(ROOT/"stage3_results"),"zip",ROOT/"results")
    files.download(zip_path)
print("Выходные файлы:",sorted(p.name for p in (ROOT/"results/demo").iterdir()))


HTML сохранён; локальный Python без IPython. Откройте results/demo/report.html в браузере.
Выходные файлы: ['comparison.csv', 'report.html', 'report.md', 'result.json', 'run.json']
